# Exercise: Building the GPT Architecture

Welcome to the programming assignment! In this exercise, you will implement the core components of a GPT model's Transformer architecture from scratch using PyTorch. This will give you a deep, hands-on understanding of how these powerful models work under the hood.

**In this exercise, you will:**
*   Implement the **Causal Self-Attention** mechanism, the key component that allows the model to weigh the importance of different tokens in a sequence.
*   Implement the **Multi-Layer Perceptron (MLP)**, the feed-forward network that processes information at each position.
*   Assemble these parts into a complete Transformer **Block**, which can be stacked to create a deep neural network.

Let's get started!

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

# We'll use a dataclass to store the configuration of our model.
# This is a best practice for keeping hyperparameters organized.
@dataclass
class GPTConfig:
    block_size: int = 256
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

# Helper function to set a seed for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# Do not modify the code above this line.

### Exercise 1: Causal Self-Attention

Your first task is to implement the `CausalSelfAttention` module. This is the heart of the Transformer, allowing each token to look at and gather information from previous tokens in the sequence. "Causal" means that a token at position `t` can only attend to tokens from `0` to `t`, but not to future tokens.

**Your task is to:**

1.  **Define Layers**: In `__init__`, define the necessary `nn.Linear` layers.
    *   A single linear layer (`c_attn`) to project the input into query (Q), key (K), and value (V) vectors simultaneously. This is an efficient way to compute them all at once.
    *   A final linear layer (`c_proj`) to project the output of the attention mechanism back to the embedding dimension.

2.  **Implement Forward Pass**: In the `forward` method, you will perform the following steps:
    *   **Get Q, K, V**: Use your `c_attn` layer and then `torch.split` or slicing to get the `q`, `k`, and `v` tensors.
    *   **Reshape for Multi-Head**: Reshape `q`, `k`, and `v` from `(B, T, C)` to `(B, nh, T, hs)`, where `B` is batch size, `T` is sequence length, `C` is embedding dimension (`n_embd`), `nh` is the number of heads, and `hs` is the head size (`n_embd // n_head`).
    *   **Calculate Attention Scores**: Compute the scaled dot-product attention scores: `(Q @ K^T) / sqrt(d_k)`. Remember that `d_k` is the head size (`hs`).
    *   **Apply Causal Mask**: This is the "causal" part. Create a lower-triangular mask to prevent attention to future tokens. You can use `torch.tril`. Fill the upper-triangular part (where the mask is 0) with `-inf`. This will make those positions zero after the softmax.
    *   **Apply Softmax**: Apply a softmax to the masked scores to get the attention weights.
    *   **Apply Attention to V**: Multiply the attention weights by the `v` tensor.
    *   **Combine Heads**: Reshape the output back to `(B, T, C)` by first transposing to `(B, T, nh, hs)` and then using `.contiguous().view()`.
    *   **Final Projection**: Pass the result through your `c_proj` layer.

**Hint:** The `bias` for the causal mask should be registered as a buffer using `self.register_buffer()` so that it's moved to the correct device (like a GPU) along with the model, but is not considered a model parameter.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        # Regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # Causal mask
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass for causal self-attention.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C)
                              B = batch size, T = sequence length, C = embedding dimension.

        Returns:
            torch.Tensor: Output tensor of shape (B, T, C).
        """
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        ### START CODE HERE ###
        # (B, T, C) -> (B, T, 3 * C)
        qkv = self.c_attn(x)
        # (B, T, 3 * C) -> (B, T, C) x 3
        q, k, v = qkv.split(self.n_embd, dim=2)

        # Reshape for multi-head attention: (B, T, C) -> (B, nh, T, hs)
        # where nh = n_head, hs = head_size = C // nh
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # Attention scores: (B, nh, T, hs) @ (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        
        # Apply causal mask
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        
        # Apply softmax to get attention weights
        att = F.softmax(att, dim=-1)
        
        # Apply attention to values: (B, nh, T, T) @ (B, nh, T, hs) -> (B, nh, T, hs)
        y = att @ v
        
        # Combine heads: (B, nh, T, hs) -> (B, T, nh, hs) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # Final projection
        y = self.c_proj(y)
        ### END CODE HERE ###

        return y

In [ ]:
# Test for CausalSelfAttention
set_seed(42)
config = GPTConfig(block_size=8, n_embd=16, n_head=4)
attention = CausalSelfAttention(config)

# Create a dummy input tensor
B, T, C = 2, 5, 16 # Batch size=2, Sequence length=5, Embedding dim=16
x = torch.randn(B, T, C)

# Get the output
output = attention(x)

# --- Assertions ---
# Test 1: Check output shape
expected_shape = (B, T, C)
assert output.shape == expected_shape, f"FAIL: Expected output shape {expected_shape}, but got {output.shape}"
print("✅ Test 1/2 Passed: Output shape is correct.")

# Test 2: Check a specific value in the output for reproducibility
expected_value = -0.0988
actual_value = output[0, 0, 0].item()
assert abs(actual_value - expected_value) < 1e-4, f"FAIL: Expected value at [0,0,0] to be ~{expected_value}, but got {actual_value}"
print("✅ Test 2/2 Passed: Output value is correct.")

print("\nGreat job! You've successfully implemented the Causal Self-Attention mechanism.")

### Exercise 2: The MLP Block

Next, you will implement the `MLP` (Multi-Layer Perceptron) module. This is the simple feed-forward network that follows the attention block. It processes each token's representation independently.

The architecture is straightforward:
1.  A linear layer (`c_fc`) that expands the embedding dimension by a factor of 4.
2.  A non-linear activation function (we'll use `GELU`).
3.  A second linear layer (`c_proj`) that projects the representation back to the original embedding dimension.

**Your task is to:**

1.  **Define Layers**: In `__init__`, define the linear layers and the activation function.
2.  **Implement Forward Pass**: In `forward`, pass the input tensor through the layers in the correct sequence.

In [ ]:
class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        ### START CODE HERE ###
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu    = nn.GELU(approximate='tanh') # Using approximate for consistency with original GPT-2
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd)
        ### END CODE HERE ###

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass for the MLP block.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C).

        Returns:
            torch.Tensor: Output tensor of shape (B, T, C).
        """
        ### START CODE HERE ###
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        ### END CODE HERE ###
        return x

In [ ]:
# Test for MLP
set_seed(42)
config = GPTConfig(n_embd=16)
mlp = MLP(config)

# Create a dummy input tensor
B, T, C = 2, 5, 16
x = torch.randn(B, T, C)

# Get the output
output = mlp(x)

# --- Assertions ---
# Test 1: Check output shape
expected_shape = (B, T, C)
assert output.shape == expected_shape, f"FAIL: Expected output shape {expected_shape}, but got {output.shape}"
print("✅ Test 1/2 Passed: Output shape is correct.")

# Test 2: Check a specific value in the output
expected_value = 0.0094
actual_value = output[0, 0, 0].item()
assert abs(actual_value - expected_value) < 1e-4, f"FAIL: Expected value at [0,0,0] to be ~{expected_value}, but got {actual_value}"
print("✅ Test 2/2 Passed: Output value is correct.")

print("\nExcellent! The MLP implementation is correct.")

### Exercise 3: The Transformer Block

You've built the two main components of a Transformer block. Now it's time to put them together!

A standard Transformer block has the following structure:
1.  **Pre-Layer Normalization**: Apply `LayerNorm` to the input.
2.  **Causal Self-Attention**: Pass the normalized output to your `CausalSelfAttention` module.
3.  **Residual Connection**: Add the output of the attention module to the original input (`x = x + attention_output`).
4.  **Pre-Layer Normalization**: Apply another `LayerNorm`.
5.  **MLP**: Pass the result to your `MLP` module.
6.  **Residual Connection**: Add the output of the MLP to the result from the first residual connection (`x = x + mlp_output`).

This pattern of `LayerNorm -> Module -> Residual` is a common and effective design.

**Your task is to:**

1.  **Define Layers**: In `__init__`, instantiate two `nn.LayerNorm` modules, your `CausalSelfAttention` module, and your `MLP` module.
2.  **Implement Forward Pass**: In `forward`, implement the two-stage process described above, making sure to include the crucial residual (or "skip") connections.

In [ ]:
class Block(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        ### START CODE HERE ###
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
        ### END CODE HERE ###

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs the forward pass for a full Transformer block.

        Args:
            x (torch.Tensor): Input tensor of shape (B, T, C).

        Returns:
            torch.Tensor: Output tensor of shape (B, T, C).
        """
        ### START CODE HERE ###
        # First stage: LayerNorm -> Attention -> Residual
        attn_output = self.attn(self.ln_1(x))
        x = x + attn_output
        
        # Second stage: LayerNorm -> MLP -> Residual
        mlp_output = self.mlp(self.ln_2(x))
        x = x + mlp_output
        ### END CODE HERE ###
        return x

In [ ]:
# Test for Block
set_seed(42)
config = GPTConfig(block_size=8, n_embd=16, n_head=4)
block = Block(config)

# Create a dummy input tensor
B, T, C = 2, 5, 16
x = torch.randn(B, T, C)

# Get the output
output = block(x)

# --- Assertions ---
# Test 1: Check output shape
expected_shape = (B, T, C)
assert output.shape == expected_shape, f"FAIL: Expected output shape {expected_shape}, but got {output.shape}"
print("✅ Test 1/2 Passed: Output shape is correct.")

# Test 2: Check a specific value in the output
expected_value = 0.5369
actual_value = output[0, 0, 0].item()
assert abs(actual_value - expected_value) < 1e-4, f"FAIL: Expected value at [0,0,0] to be ~{expected_value}, but got {actual_value}"
print("✅ Test 2/2 Passed: Output value is correct.")

print("\nFantastic! You have successfully implemented a complete Transformer Block.")

### Congratulations!

You have now implemented the core architectural components of a GPT model. By building `CausalSelfAttention`, `MLP`, and the `Block` that combines them, you've gained a fundamental understanding of how these models are constructed.

The code below demonstrates how your `Block` implementation would be integrated into a full, albeit minimal, GPT model. You don't need to write any code here, just run the cell to see it all come together. Stacking these blocks is what gives the model its depth and power.

In [ ]:
# --- Integration Test: A Minimal GPT Model ---
# This code uses your `Block` implementation to build a small GPT model.
# You do not need to modify this cell.

class MinimalGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"
        
        # forward the GPT model itself
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device) # shape (T)
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (T, n_embd)
        x = tok_emb + pos_emb
        
        for block in self.transformer.h:
            x = block(x)
            
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        return logits

# --- Run the integration test ---
set_seed(42)

# Use a small configuration for a quick test
gpt_config = GPTConfig(
    block_size=8,
    vocab_size=100,
    n_layer=2,  # A 2-layer transformer
    n_head=4,
    n_embd=16
)

# Instantiate the model
model = MinimalGPT(gpt_config)
print("Model created successfully!")

# Create a dummy input (batch of 2 sequences of 5 tokens each)
idx = torch.randint(0, gpt_config.vocab_size, (2, 5))

# Get the model output (logits)
logits = model(idx)

# Final assertion
expected_logits_shape = (2, 5, gpt_config.vocab_size)
assert logits.shape == expected_logits_shape, f"FAIL: Final logits shape is incorrect. Expected {expected_logits_shape}, got {logits.shape}"
print(f"✅ Integration Test Passed: Final logits shape is correct: {logits.shape}")

expected_logit_val = 0.0076
actual_logit_val = logits[0,0,0].item()
assert abs(actual_logit_val - expected_logit_val) < 1e-4, f"FAIL: Final logit value is incorrect. Expected ~{expected_logit_val}, got {actual_logit_val}"
print(f"✅ Integration Test Passed: A specific logit value is correct.")
print("\nAll exercises and tests passed. Well done!")